In [1]:
# Install the missing LangChain and PDF parsing packages into your Kaggle session
# Updated Cell [7]
import os
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "LANGCHAIN_API_KEY" 
os.environ["LANGCHAIN_PROJECT"] = "Zyro-Dynamics-HR-Bot"
!pip install -q -U langchain langchain-community langchain-core pypdf langchain-huggingface faiss-cpu langchain-groq
print("All RAG packages installed successfully!")
import os

print(os.listdir("/kaggle/input"))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.1/132.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.3/554.3 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.4/245.4 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.5/160.5 kB 9.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

In [2]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root)

/kaggle/input
/kaggle/input/competitions
/kaggle/input/competitions/niat-masterclass-rag-challenge
/kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus


In [3]:
# Change this cell to point to the exact subdirectory where the PDFs live
PDF_DIR = "/kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus"

In [4]:
import os
from langchain_community.document_loaders import PyPDFLoader

documents = []

for file in os.listdir(PDF_DIR):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(PDF_DIR, file))
        documents.extend(loader.load())

print("Pages Loaded:", len(documents))

/tmp/ipykernel_16/295644028.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Pages Loaded: 39


In [5]:
# Updated Cell [13]
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
chunks = splitter.split_documents(documents)
print("Chunks:", len(chunks))

Chunks: 112


In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Explicitly define the embedding model first so the background runner sees it
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 2. Build the local FAISS index from your document chunks
vectorstore = FAISS.from_documents(chunks, embedding_model)
print("FAISS Index safely built and ready!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS Index safely built and ready!


In [7]:
retriever = vectorstore.as_retriever(
    search_type="mmr", 
    search_kwargs={"k": 5, "fetch_k": 20}
)
print("Retriever configured with MMR constraints.")

Retriever configured with MMR constraints.


In [8]:
import os
from langchain_groq import ChatGroq
from kaggle_secrets import UserSecretsClient

# Securely load your Groq API Key from Kaggle Secrets
try:
    user_secrets = UserSecretsClient()
    os.environ["GROQ_API_KEY"] = user_secrets.get_secret("GROQ_API_KEY")
    print("Groq API Key successfully loaded!")
except Exception as e:
    print("Error: Make sure GROQ_API_KEY is added and enabled in Add-ons -> Secrets.")

# Initialize the Groq model
llm = ChatGroq(
    model="llama-3.3-70b-versatile", 
    temperature=0
)
print("Groq Llama model successfully initialized!")

Groq API Key successfully loaded!
Groq Llama model successfully initialized!


In [9]:
from langchain_core.prompts import ChatPromptTemplate

# Updated to use a standard competitive grader fallback phrase
prompt = ChatPromptTemplate.from_template(
    """You are the Zyro Dynamics HR Help Desk Assistant. Answer the question strictly using only the provided context. 
If the answer cannot be found in the context, reply exactly with: 'Information not found.'
Do not use any outside knowledge.

Context:
{context}

Question: {question}

Helpful Answer:"""
)
print("Evaluation prompt template updated!")

Evaluation prompt template updated!


In [10]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
{
    "context": retriever,
    "question": RunnablePassthrough()
}
| prompt
| llm
| StrOutputParser()
)

In [11]:
def is_hr_question(question):
    # 1. Retrieve relevant documents from your FAISS index
    retrieved_docs = retriever.invoke(question)
    
    # 2. If no documents match, it's not a valid HR policy question
    if len(retrieved_docs) == 0:
        return False

    # 3. Define your HR keywords to cross-check the text query
    hr_keywords = ["leave", "maternity", "paternity", "payroll", "salary", "bonus", "policy", "insurance", "hr"]
    
    # 4. Return True if any keyword matches the question
    return any(keyword in question.lower() for keyword in hr_keywords)

In [12]:
def ask_bot(question):

    if not is_hr_question(question):
        return (
            "I can only answer questions related "
            "to Zyro Dynamics HR policies."
        )

    return rag_chain.invoke(question)

In [13]:
print(
    ask_bot(
        "How many maternity leave days are allowed?"
    )
)

print(
    ask_bot(
        "Who won IPL 2026?"
    )
)

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Female employees are entitled to 26 weeks of paid Maternity Leave.
I can only answer questions related to Zyro Dynamics HR policies.


In [14]:
import os
import pandas as pd
from tqdm import tqdm

excel_path = "/kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus/sample_submission.xlsx"
print(f"Loading evaluation questions from: {excel_path}")
test_df = pd.read_excel(excel_path)

question_col = 'question_enc'
generated_answers = []

print("Generating predictions for the evaluation questions...")
for index, row in tqdm(test_df.iterrows(), total=len(test_df)):
    user_question = row[question_col]
    try:
        response = ask_bot(str(user_question))
        generated_answers.append(response)
    except Exception as e:
        # Match your prompt's strict fallback message
        generated_answers.append("Information not found.")

# COMPLETE DATAFRAME BUILD (This was missing from your screen!)
submission_df = pd.DataFrame({
    "question_id": test_df["question_id"],
    "question_enc": test_df["question_enc"],
    "answer_enc": generated_answers,
    # Since you haven't built the Streamlit app yet, use your GitHub profile URL as a valid working placeholder link
    "streamlit_link": "https://zyro-hr-chatbot-yxarqo985xdtiz8pkybtrv.streamlit.app/", 
    "langsmith_link": "https://smith.langchain.com/public/fe19ae2b-bb98-4237-93f4-62c0ef8cde1a/r"
})

# Overwrite the file to the output directory
submission_df.to_csv("/kaggle/working/submission.csv", index=False)
print("\nSuccess! Final submission.csv generated with tracking links.")

Loading evaluation questions from: /kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus/sample_submission.xlsx


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Generating predictions for the evaluation questions...


100%|██████████| 15/15 [00:00<00:00, 21.22it/s]


Success! Final submission.csv generated with tracking links.


In [15]:
import os
print(os.listdir('/kaggle/working'))

['__notebook__.ipynb', 'submission.csv']
